## Assignment 11

### Imports

In [ ]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn

# For easy copying
"""../../MainProject/Assignment11/data/kinect_good_preprocessed_not_cut_start_stop"""

### Step 1: Split CSV Files

In [ ]:
def split_csv_files(datafolder: str, train_size: float = 0.8, val_size: float = 0.1, test_size: float = 0.1, random_seed: int = 42):
    """Split CSV files into train/val/test sets"""
    csv_files = [f for f in os.listdir(datafolder) if f.endswith('.csv')]
    
    train_val, test = train_test_split(
        csv_files, 
        test_size=test_size, 
        random_state=random_seed
    )
    
    train, val = train_test_split(
        train_val,
        train_size=train_size/(train_size+val_size),
        random_state=random_seed
    )
    
    print(f"Train: {len(train)} files, Val: {len(val)} files, Test: {len(test)} files")
    return train, val, test

### Step 2: Create Time Series Dataset Class for LSTM

In [34]:
class TimeSeriesDataset(Dataset):
    """Custom Dataset for LSTM time series prediction"""
    
    def __init__(self, file_list, data_folder, sequence_length=10, target_column='target'):
        """
        Args:
            file_list: List of CSV file names
            data_folder: Path to folder containing CSV files
            sequence_length: Number of time steps to use as input (lookback window)
            target_column: Name of the column to predict
        """
        self.sequences = []
        self.targets = []
        
        for file_name in file_list:
            # Load CSV
            file_path = os.path.join(data_folder, file_name)
            df = pd.read_csv(file_path)
            
            # Assume last column is target, or specify target_column
            if target_column not in df.columns:
                # Use last column as target if specified column not found
                target_col = df.columns[-1]
            else:
                target_col = target_column
            
            # Separate features and target
            feature_cols = [col for col in df.columns if col != target_col]
            features = df[feature_cols].values
            target = df[target_col].values
            
            # Create sequences
            for i in range(len(features) - sequence_length):
                seq_features = features[i:i+sequence_length]
                seq_target = target[i+sequence_length]  # Predict next value
                
                self.sequences.append(seq_features)
                self.targets.append(seq_target)
        
        # Convert to numpy arrays
        self.sequences = np.array(self.sequences, dtype=np.float32)
        self.targets = np.array(self.targets, dtype=np.float32)
        
    def __len__(self):
        return len(self.sequences)
    
    def __getitem__(self, idx):
        return torch.tensor(self.sequences[idx]), torch.tensor(self.targets[idx])